# 第8回　検定の多重性問題
## ―― たくさん試せば、偶然の「有意」はいくらでも出る

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

統計学Ⅰで仮説検定とp値を学んだ。Ⅱでは、その **誤用** の代表格 ―― 検定をたくさん繰り返すことで生まれる偽の発見 ―― を暴く。▶ を上から押そう。

有名な風刺漫画（xkcd "Significant"）：「**緑のゼリービーンズはニキビの原因だ！**（p<0.05）」。20色を1つずつ調べたら、緑だけ偶然 p<0.05 になった、というオチだ。これを自分の手で再現する。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. 効果ゼロでも、20色のうち1つは「有意」になる

20色のゼリービーンズ。**どの色もニキビとは本当は無関係**（効果ゼロ）だとして、各色について「食べた群 vs 食べない群」でニキビの数を t 検定する。

効果はゼロなのだから、有意（p<0.05）は出ないはず？　予想を決めてから ▶。

In [ ]:
rng = np.random.default_rng(8)
色 = ["赤","橙","黄","緑","青","藍","紫","茶","桃","水",
      "灰","白","黒","金","銀","碧","朱","紺","翠","杏"]
n = 30
print("各色の検定結果（本当はどの色も効果ゼロ）：")
有意な色 = []
for c in 色:
    群A = rng.normal(10, 3, n)   # 食べた群のニキビ数
    群B = rng.normal(10, 3, n)   # 食べない群（同じ分布＝効果ゼロ）
    p = stats.ttest_ind(群A, 群B).pvalue
    印 = " ★有意！" if p < 0.05 else ""
    if p < 0.05:
        有意な色.append(c)
    print(f"  {c}色： p = {p:.3f}{印}")
print(f"\n→ 効果ゼロなのに、有意になった色： {有意な色 if 有意な色 else 'なし'}")

**本当はどの色も無関係なのに、有意な色が出てしまう。** これを「緑はニキビの原因だ」と報告したら、完全な誤りだ。

なぜか。1回の検定で偶然有意になる確率は5%。だが **20回もやれば**、どれか1つが偶然5%を引く確率はぐっと上がる。

---
## 2. 偽陽性が積み上がる ―― ファミリーワイズ・エラー

効果ゼロのとき、1回の検定で「有意になってしまう」確率は $\alpha=0.05$。
独立に $m$ 回検定して、**少なくとも1回は偽陽性が出る**確率は：

$$1-(1-\alpha)^{m}$$

$m=20$ なら $1-0.95^{20}\approx0.64$。**6割を超える**。実際に1000回シミュレーションして確かめよう。

In [ ]:
def 少なくとも1つ有意になる割合(m, 試行=2000, α=0.05):
    回数 = 0
    for _ in range(試行):
        ps = [stats.ttest_ind(rng.normal(10,3,30), rng.normal(10,3,30)).pvalue for _ in range(m)]
        if min(ps) < α:
            回数 += 1
    return 回数 / 試行

for m in [1, 5, 10, 20]:
    実測 = 少なくとも1つ有意になる割合(m)
    理論 = 1 - (1 - 0.05) ** m
    print(f"m={m:>2}回検定 → 少なくとも1つ偽陽性： 実測 {実測:.1%} / 理論 {理論:.1%}")

検定の回数が増えるほど、偽陽性に出くわす確率が跳ね上がる。
**「たくさん調べて、有意だったものだけ報告する」**――これは発見ではなく、偶然を拾っているだけだ。研究不正と紙一重のこの行為を **p-hacking** と呼ぶ。

---
## 3. 対策 ―― 多重比較補正（ボンフェローニ）

$m$ 回検定するなら、基準を厳しくすればよい。最も簡単な **ボンフェローニ補正** は、有意水準を $\alpha/m$ にする（20回なら $0.05/20=0.0025$）。

補正なし／ありで、偽陽性率がどう変わるか比べよう。

In [ ]:
def 偽陽性率(補正, m=20, 試行=2000, α=0.05):
    基準 = α / m if 補正 else α
    回数 = sum(
        min(stats.ttest_ind(rng.normal(10,3,30), rng.normal(10,3,30)).pvalue for _ in range(m)) < 基準
        for _ in range(試行)
    )
    return 回数 / 試行

なし = 偽陽性率(補正=False)
あり = 偽陽性率(補正=True)
print(f"20回検定で『少なくとも1つ偽陽性』が出る割合")
print(f"  補正なし（各回 α=0.05）　　　： {なし:.1%}")
print(f"  ボンフェローニ補正（α=0.0025）： {あり:.1%}  ← 5%前後に戻る")

plt.figure(figsize=(6, 4))
plt.bar(["補正なし", "ボンフェローニ補正"], [なし, あり], color=["#e8503a", "#3949ab"])
plt.axhline(0.05, ls="--", color="gray", label="目標 5%")
plt.ylabel("偽陽性が出た割合"); plt.ylim(0, 0.8)
plt.title("多重比較補正で、偽陽性を抑えられる")
for i, v in enumerate([なし, あり]):
    plt.text(i, v + 0.02, f"{v:.0%}", ha="center")
plt.legend(); plt.show()

補正すれば偽陽性率は目標の5%前後に戻る。検定をたくさんするなら、**基準を厳しくする**のが筋だ（ボンフェローニのほか、より緩やかな FDR 制御などもある）。

だがもっと大事なのは ―― **そもそも、有意が出るまで検定を繰り返さないこと**。次の対策がそれだ。

---
## 4. もっと根本の対策 ―― 事前登録

p-hacking には、補正だけでは防げない巧妙な形がある。

- **HARKing**：結果を見てから「最初からこの仮説でした」と後付けする
- **チェリーピッキング**：有意だったサブグループ・指標だけを報告する
- **optional stopping**：有意になるまでデータを足し続ける

これらの根本対策が **事前登録（pre-registration）** ―― データを取る **前に**、「どの仮説を・どの方法で検定するか」を公開して固定してしまう。後出しを封じるのだ。

> 💬 **統計的誤用を見抜く目**
> 
> 「有意差が出ました（p<0.05）」と聞いたら、まず問う ―― **『それ、何回検定したうちの1つ？』『仮説は先に決めてあった？』**。多くの“発見”は、たくさん試した中のまぐれ当たりだ。これを疑えることが、第9回（効果量とp値批判）へとつながる。


---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| 多重性問題 | 検定を $m$ 回繰り返すと、偽陽性確率は $1-(1-\alpha)^m$ に膨らむ（20回で約64%） |
| p-hacking | 有意が出るまで試す／有意なものだけ報告する。偶然を発見と偽る |
| ボンフェローニ補正 | 有意水準を $\alpha/m$ に厳しくして偽陽性を抑える |
| 事前登録 | データを取る前に仮説と方法を固定し、後出しを封じる |

- 効果ゼロでも、20回検定すれば6割超で「有意」が出る。
- 「有意差が出た」と聞いたら **「何回試したうちの1つ？」** を問う。

> **課題（Moodle）**：多重検定シナリオの判断（自動採点）＋「この研究のp値はなぜ信用できないか」の批判的記述。詳しくはMoodleの第8回課題を見ること。

> **次回予告**：第9回「効果量とp値批判」。今度は逆に「n＝100万なら、無意味な差でも p<0.001 になる」 ―― 有意であることと、重要であることは、まったく別の話だ。